### Евсеев ИУ5-64Б Семинар 10

## Модель лифта в здании
- В 5-этажном здании есть один лифт, вмещающий не более 6 человек.
- На вход этого здания на 1-м этаже поступает поток посетителей с интервалом от 1 до 20.
- Первый этаж – технический, и посетители на нем не задерживаются, а направляются на 2,3,4,5-й этажи, пользуясь лифтом.
- Посетитель, попав на свой этаж, находится на нем в течение случайного времени в интервале от 50 до 90.
- После этого он направляется к лифту, на нем опускается на 1-й этаж и покидает здание.
- Длительность перемещения лифта на один этаж равно 12. Длительность остановки лифта на этаже равно 10.

In [1]:
# -*- coding: utf-8 -*-
import random
import simpy
#
RANDOM_SEED = 526
MAX_LEVEL = 5

In [2]:
class Lift():
    def __init__(self, name, env):
        self.env = env  # ссылка на среду
        self.name = name  # имя лифта
        self.maxx = 6  # макс.емкость лифта
        self.curr_floor = 0  # этаж лифта
        self.going_up = True  # лифт едет вверх?
        self.Lift1Time = 12  # время между этажами
        self.Lift1Stop = 10  # время на этаже
        self.liftcab = simpy.Resource(self.env, capacity=self.maxx)  # ресурс
        # создаем словарь событий - этажи
        self.eventslift = {(i+1): env.event() for i in range(MAX_LEVEL)}

    def num_generator(self):
      while True:  # -1-2-3-4-3-2-
        i = 1
        while i <= MAX_LEVEL:
          yield int(i); i += 1
        i = MAX_LEVEL-1
        while i > 1:
          yield int(i); i -= 1

    def run(self):
        global  clients

        stage = self.num_generator()  # генератор - источник этажа
        yield self.env.timeout(5)  # подождем первых ездоков
        while len(clients) > 0:
            self.curr_floor = next(stage)  # лифт на этаже -использование генератора
            if (self.curr_floor in [1, MAX_LEVEL]):
                self.going_up = (self.curr_floor==1)  # лифт едет вверх?
            print(self.env.now, ":Ha_",self.curr_floor, " /",self.liftcab.count) # лифт приехал на этаж
            self.eventslift[self.curr_floor].succeed()  # создаем событие - на этаже!
            yield self.env.timeout(self.Lift1Stop)  # время остановки на этаже
            self.eventslift[self.curr_floor] = self.env.event()  # пересоздаем событие
            print(self.env.now, ":C_",self.curr_floor, " /",self.liftcab.count) # лифт уехал с этажа
            self.curr_floor = 0  # лифт не на этаже
            yield self.env.timeout(self.Lift1Time)  # время переезда на след.этаж
        else:
            print("That's all, folks! =", self.env.now)

In [3]:
class Client():
    def __init__(self, name, env):
        self.office_walk = random.randint(50,90) # время пребывания на этаже
        self.name = name   # имя клиента
        self.lift = env.lift  # ссылка на лифт
        self.env = env   # ссылка на среду
        self.gentime = self.env.now
        self.curr_floor = 1   # с какого этажа
        self.target_floor = random.randint(2,MAX_LEVEL)  # на какой этаж
        self.inside_ = False  # флаг - в лифте
        self.walking = False  # флаг - гуляю на этаже

    def run(self):
        global  clients
        while True:
            if not self.walking:
                yield self.lift.eventslift[self.curr_floor] # жду лифт на моем этаже
                #
                if (self.lift.going_up == (self.target_floor > self.curr_floor)) and \
                   (self.lift.liftcab.count < self.lift.maxx) :  # если лифт в нужном направлении и есть место
                    enterin = self.lift.liftcab.request() # занимаем ресурс
                    yield enterin
                    #
                    self.inside_ = True # клиент в лифте
                    yield self.env.timeout(1) # вход в лифт
                    print(self.env.now,">>",self.name, "!_",self.curr_floor, "!^",self.target_floor)
                    yield self.lift.eventslift[self.target_floor] # едем до нужного этажа
                    self.lift.liftcab.release(enterin)     # освобождаем ресурс
                    #
                    self.inside_ = False # клиент вышел из лифта
                    self.curr_floor = self.lift.curr_floor
                    self.walking = True # будет прогулка по делам в офисе
                    self.target_floor = 1 # и возвращение на 1 этаж
                    print(self.env.now,"<<",self.name, "!_",self.curr_floor, "!^",self.target_floor)
                    yield self.env.timeout(10)
                    if self.curr_floor==1:     # вернулся на 1й этаж?
                        clients.remove(self.name) # удаляем из числа клиентов
                        print(f"*{self.name}*: {self.env.now-self.gentime}: в офисе осталось {len(clients)} чел")
                        return
                else:
                    yield self.env.timeout(1) # надо подождать место в лифте
            else:
                yield self.env.timeout(self.office_walk) # прогулка по делам в офисе
                self.walking = False # прогулка закончена

In [4]:
def client_gen(env, qty_passengers):
  def char_generator():
      while True:  # ABC..Z+abc..z+0..9
        i = 65
        while i <= 90:
          yield chr(i); i += 1
        i = 97
        while i <= 122:
          yield chr(i); i += 1
        i = 57
        while i >= 48:
          yield chr(i); i -= 1  

  global clients
  # создаем пассажиров
  nam = char_generator()
  for _ in range(qty_passengers):
    man = Client(next(nam), env)
    clients.append(man.name)
    env.process(man.run())
    yield env.timeout(random.randint(1,20))

In [5]:
env = simpy.Environment()
# список всех пассажиров
clients = []
#random.seed(RANDOM_SEED)

# создаем лифт
env.lift = Lift("_Lift_", env)
# подключаем процесс лифта
env.process(env.lift.run())
# подключаем процесс генерации клиентов
env.process(client_gen(env, 60))
# запускаем модель
env.run(until=2000)

5 :Ha_ 1  / 0
6 >> A !_ 1 !^ 5
15 :C_ 1  / 1
27 :Ha_ 2  / 1
37 :C_ 2  / 1
49 :Ha_ 3  / 1
59 :C_ 3  / 1
71 :Ha_ 4  / 1
81 :C_ 4  / 1
93 :Ha_ 5  / 1
93 << A !_ 5 !^ 1
103 :C_ 5  / 0
115 :Ha_ 4  / 0
125 :C_ 4  / 0
137 :Ha_ 3  / 0
147 :C_ 3  / 0
159 :Ha_ 2  / 0
169 :C_ 2  / 0
181 :Ha_ 1  / 0
182 >> B !_ 1 !^ 4
182 >> C !_ 1 !^ 5
182 >> D !_ 1 !^ 3
182 >> E !_ 1 !^ 2
182 >> F !_ 1 !^ 3
182 >> G !_ 1 !^ 3
191 :C_ 1  / 6
203 :Ha_ 2  / 6
203 << E !_ 2 !^ 1
213 :C_ 2  / 5
225 :Ha_ 3  / 5
225 << D !_ 3 !^ 1
225 << F !_ 3 !^ 1
225 << G !_ 3 !^ 1
235 :C_ 3  / 2
247 :Ha_ 4  / 2
247 << B !_ 4 !^ 1
257 :C_ 4  / 1
269 :Ha_ 5  / 1
269 << C !_ 5 !^ 1
270 >> A !_ 5 !^ 1
279 :C_ 5  / 1
291 :Ha_ 4  / 1
301 :C_ 4  / 1
313 :Ha_ 3  / 1
314 >> G !_ 3 !^ 1
314 >> F !_ 3 !^ 1
314 >> D !_ 3 !^ 1
323 :C_ 3  / 4
335 :Ha_ 2  / 4
336 >> E !_ 2 !^ 1
345 :C_ 2  / 5
357 :Ha_ 1  / 5
357 << A !_ 1 !^ 1
357 << G !_ 1 !^ 1
357 << F !_ 1 !^ 1
357 << D !_ 1 !^ 1
357 << E !_ 1 !^ 1
358 >> Q !_ 1 !^ 3
358 >> a !_ 1 !^ 2
358 >> 

*Задание для самостоятельной работы:*
1) записать в журнал время ожидания лифта пассажирами в очередях на этажах
2) посчитать количество поездок лифта с 1 на 4 и обратно на 1 за время моделирования
3) скорректировать выход клиентов из модели на 1м этаже - без задержки
4) записать в журнал длительность жизни клиентов в модели

In [6]:
# -*- coding: utf-8 -*-
import random
import simpy

RANDOM_SEED = 526
MAX_LEVEL = 5

class Lift():
    def __init__(self, name, env):
        self.env = env  # ссылка на среду
        self.name = name  # имя лифта
        self.maxx = 6  # макс.емкость лифта
        self.curr_floor = 0  # этаж лифта
        self.going_up = True  # лифт едет вверх?
        self.Lift1Time = 12  # время между этажами
        self.Lift1Stop = 10  # время на этаже
        self.liftcab = simpy.Resource(self.env, capacity=self.maxx)  # ресурс
        # создаем словарь событий - этажи
        self.eventslift = {(i+1): env.event() for i in range(MAX_LEVEL)}
        
        # задания для самостоятельной работы
        self.trips_1_to_4 = 0
        self.trips_4_to_1 = 0
        self.prev_floor = 1

    def num_generator(self):
        while True:  # -1-2-3-4-3-2-
            i = 1
            while i <= MAX_LEVEL:
                yield int(i)
                i += 1
            i = MAX_LEVEL - 1
            while i > 1:
                yield int(i)
                i -= 1

    def run(self):
        global clients, wait_log, life_log

        stage = self.num_generator()  # генератор - источник этажа
        yield self.env.timeout(5)  # подождем первых ездоков
        while True:
            self.curr_floor = next(stage)  # лифт на этаже -использование генератора
            
            # задания для самостоятельной работы (пункт 2)
            if self.prev_floor == 1 and self.curr_floor == 4:
                self.trips_1_to_4 += 1
            if self.prev_floor == 4 and self.curr_floor == 1:
                self.trips_4_to_1 += 1
            self.prev_floor = self.curr_floor
            
            if (self.curr_floor in [1, MAX_LEVEL]):
                self.going_up = (self.curr_floor == 1)  # лифт едет вверх?
            print(self.env.now, ":Ha_", self.curr_floor, " /", self.liftcab.count)  # лифт приехал на этаж
            self.eventslift[self.curr_floor].succeed()  # создаем событие - на этаже!
            yield self.env.timeout(self.Lift1Stop)  # время остановки на этаже
            self.eventslift[self.curr_floor] = self.env.event()  # пересоздаем событие
            print(self.env.now, ":C_", self.curr_floor, " /", self.liftcab.count)  # лифт уехал с этажа
            self.curr_floor = 0  # лифт не на этаже
            yield self.env.timeout(self.Lift1Time)  # время переезда на след.этаж
            
            # задания для самостоятельной работы
            if len(clients) == 0:
                print("That's all, folks! =", self.env.now)
                print("Поездок 1->4:", self.trips_1_to_4)
                print("Поездок 4->1:", self.trips_4_to_1)
                break

class Client():
    def __init__(self, name, env):
        self.office_walk = random.randint(50, 90)  # время пребывания на этаже
        self.name = name   # имя клиента
        self.lift = env.lift  # ссылка на лифт
        self.env = env   # ссылка на среду
        self.gentime = self.env.now
        self.curr_floor = 1   # с какого этажа
        self.target_floor = random.randint(2, MAX_LEVEL)  # на какой этаж
        self.inside_ = False  # флаг - в лифте
        self.walking = False  # флаг - гуляю на этаже
        
        # задания для самостоятельной работы (пункт 1)
        self.wait_start = None

    def run(self):
        global clients, wait_log, life_log
        
        while True:
            if not self.walking:
                # задания для самостоятельной работы (пункт 1)
                self.wait_start = self.env.now
                yield self.lift.eventslift[self.curr_floor]  # жду лифт на моем этаже
                
                if (self.lift.going_up == (self.target_floor > self.curr_floor)) and \
                   (self.lift.liftcab.count < self.lift.maxx):  # если лифт в нужном направлении и есть место
                    
                    # задания для самостоятельной работы (пункт 1)
                    wait_time = self.env.now - self.wait_start
                    wait_log.append((self.name, self.curr_floor, wait_time))
                    
                    enterin = self.lift.liftcab.request()  # занимаем ресурс
                    yield enterin
                    
                    self.inside_ = True  # клиент в лифте
                    yield self.env.timeout(1)  # вход в лифт
                    print(self.env.now, ">>", self.name, "!_", self.curr_floor, "!^", self.target_floor)
                    yield self.lift.eventslift[self.target_floor]  # едем до нужного этажа
                    self.lift.liftcab.release(enterin)  # освобождаем ресурс
                    
                    self.inside_ = False  # клиент вышел из лифта
                    self.curr_floor = self.lift.curr_floor
                    self.walking = True  # будет прогулка по делам в офисе
                    self.target_floor = 1  # и возвращение на 1 этаж
                    print(self.env.now, "<<", self.name, "!_", self.curr_floor, "!^", self.target_floor)
                    
                    # задания для самостоятельной работы (пункт 3 - выход без задержки)
                    if self.curr_floor == 1:  # вернулся на 1й этаж?
                        # задания для самостоятельной работы (пункт 4)
                        life_time = self.env.now - self.gentime
                        life_log.append((self.name, life_time))
                        clients.remove(self.name)  # удаляем из числа клиентов
                        print(f"*{self.name}*: {self.env.now-self.gentime}: в офисе осталось {len(clients)} чел")
                        return
                else:
                    yield self.env.timeout(1)  # надо подождать место в лифте
            else:
                yield self.env.timeout(self.office_walk)  # прогулка по делам в офисе
                self.walking = False  # прогулка закончена

def client_gen(env, qty_passengers):
    def char_generator():
        while True:  # ABC..Z+abc..z+0..9
            i = 65
            while i <= 90:
                yield chr(i)
                i += 1
            i = 97
            while i <= 122:
                yield chr(i)
                i += 1
            i = 57
            while i >= 48:
                yield chr(i)
                i -= 1

    global clients
    # создаем пассажиров
    nam = char_generator()
    for _ in range(qty_passengers):
        man = Client(next(nam), env)
        clients.append(man.name)
        env.process(man.run())
        yield env.timeout(random.randint(1, 20))

# задания для самостоятельной работы
env = simpy.Environment()
clients = []
wait_log = []
life_log = []

# создаем лифт
env.lift = Lift("_Lift_", env)
# подключаем процесс лифта
env.process(env.lift.run())
# подключаем процесс генерации клиентов
env.process(client_gen(env, 60))
# запускаем модель
env.run()

# задания для самостоятельной работы (вывод журналов)
print("\n--- Журнал ожидания ---")
for entry in wait_log:
    print(f"Клиент {entry[0]}, этаж {entry[1]}, время ожидания: {entry[2]:.2f}")

print("\n--- Журнал длительности жизни ---")
for entry in life_log:
    print(f"Клиент {entry[0]}, время в системе: {entry[1]:.2f}")

5 :Ha_ 1  / 0
6 >> A !_ 1 !^ 5
6 >> B !_ 1 !^ 3
6 >> C !_ 1 !^ 3
12 >> D !_ 1 !^ 4
15 :C_ 1  / 4
27 :Ha_ 2  / 4
37 :C_ 2  / 4
49 :Ha_ 3  / 4
49 << B !_ 3 !^ 1
49 << C !_ 3 !^ 1
59 :C_ 3  / 2
71 :Ha_ 4  / 2
71 << D !_ 4 !^ 1
81 :C_ 4  / 1
93 :Ha_ 5  / 1
93 << A !_ 5 !^ 1
103 :C_ 5  / 0
115 :Ha_ 4  / 0
125 :C_ 4  / 1
126 >> D !_ 4 !^ 1
137 :Ha_ 3  / 1
138 >> C !_ 3 !^ 1
138 >> B !_ 3 !^ 1
147 :C_ 3  / 3
159 :Ha_ 2  / 3
169 :C_ 2  / 3
181 :Ha_ 1  / 3
181 << D !_ 1 !^ 1
*D*: 170: в офисе осталось 18 чел
181 << C !_ 1 !^ 1
*C*: 179: в офисе осталось 17 чел
181 << B !_ 1 !^ 1
*B*: 180: в офисе осталось 16 чел
182 >> E !_ 1 !^ 4
182 >> F !_ 1 !^ 4
182 >> G !_ 1 !^ 5
182 >> O !_ 1 !^ 4
182 >> P !_ 1 !^ 4
182 >> Q !_ 1 !^ 5
191 :C_ 1  / 6
203 :Ha_ 2  / 6
213 :C_ 2  / 6
225 :Ha_ 3  / 6
235 :C_ 3  / 6
247 :Ha_ 4  / 6
247 << E !_ 4 !^ 1
247 << F !_ 4 !^ 1
247 << O !_ 4 !^ 1
247 << P !_ 4 !^ 1
257 :C_ 4  / 2
269 :Ha_ 5  / 2
269 << G !_ 5 !^ 1
269 << Q !_ 5 !^ 1
270 >> A !_ 5 !^ 1
279 :C_ 5  / 1
291